# EURIBOR BASIS FAIR VALUE MODEL:

### Import Packages:

In [ ]:
import blpapi
import matplotlib.pyplot as plt
import pdblp
import datetime as dt
import pandas as pd
import numpy as np
import math
import statsmodels.tsa.stattools as ts
import statsmodels.api as sm
import seaborn as sns
from sklearn import linear_model
import functools
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.regression.rolling import RollingOLS
from typing import Optional, Union


### Bloomberg Api Connection

In [ ]:
con = pdblp.BCon(debug=False, port=8194, timeout=50000)

con.start()

### Auxiliary Function:

In [ ]:
def _parse_tenor_to_suffix(tenor: Union[str, int], *, code_suffix_override: Optional[str] = None) -> str:
    """
    Helper function to normalize tenor input into a Bloomberg suffix (e.g. "25Y" -> "25").
    """
    if code_suffix_override:
        return str(code_suffix_override)

    if isinstance(tenor, int):
        if tenor <= 0:
            raise ValueError("Tenor (years) must be positive")
        return str(tenor)

    if isinstance(tenor, str):
        s = tenor.strip().upper()
        if s.endswith("Y"):   # e.g. "25Y" -> "25"
            return s[:-1]
        if s.isdigit():       # e.g. "10" -> "10"
            return s
        raise ValueError(f"Unsupported tenor format: {tenor}")

    raise TypeError("Tenor must be int or str")


def make_basis_ticker(
    basis_type: str,
    tenor: Union[str, int],
    *,
    code_suffix_override: Optional[str] = None,
    generic: bool = True
) -> str:
    basis_type = basis_type.lower().strip()
    suffix = _parse_tenor_to_suffix(tenor, code_suffix_override=code_suffix_override)

    if basis_type == "estr":
        root = f"EESW{suffix}"
        return f"{root} BGN Curncy" if generic else root

    elif basis_type in ["3s6s", "euribor3m6m"]:
        root = f"EUBSV{suffix}"
        return f"{root} CMPN Curncy" if generic else root

    else:
        raise ValueError(f"Unknown basis_type '{basis_type}'. Use 'estr' or '3s6s'.")
        

def bbg_download_clean_df(ticker_list, fields, start_date, end_date, period=None):
    start_date, end_date = start_date.strftime("%Y%m%d"), end_date.strftime("%Y%m%d")
    if period is None:
        tmp = con.bdh(ticker_list, fields, start_date, end_date)
    else:
        tmp = con.bdh(ticker_list, fields, start_date, end_date, elms=[('periodicitySelection', period)])

    tmp = tmp[ticker_list]
    tmp.columns = tmp.columns.droplevel(1)
    tmp = tmp.sort_values(by='date', ascending=False)
    return tmp


def forward_series_from_timeseries(df, start_tenor, end_tenor, compounding="annual"):

    def _parse_tenor_to_years(x):
        s = str(x).upper().strip()
        if s.endswith("M"):
            return float(s[:-1]) / 12.0
        elif s.endswith("Y"):
            return float(s[:-1])
        else:
            return float(s)

    T1 = _parse_tenor_to_years(start_tenor)
    T2 = _parse_tenor_to_years(end_tenor)

    tenors = [c for c in df.columns if c != 'Date']
    tenor_years = [_parse_tenor_to_years(t) for t in tenors]
    sort_idx = np.argsort(tenor_years)
    tenors = [tenors[i] for i in sort_idx]
    tenor_years = [tenor_years[i] for i in sort_idx]

    results = []

    for _, row in df.iterrows():
        curve = row[tenors].values.astype(float)

        # interpolazione dei tassi spot per T1 e T2
        R1 = np.interp(T1, tenor_years, curve)
        R2 = np.interp(T2, tenor_years, curve)

        if compounding == "simple":
            fwd = ((1 + R2 * T2) / (1 + R1 * T1) - 1) / (T2 - T1)
        else:
            fwd = ((1 + R2) ** T2 / (1 + R1) ** T1) ** (1 / (T2 - T1)) - 1

        results.append(fwd)

    out = pd.DataFrame({
        "Date": df.index,
        f"Fwd_{str(int(T1))}Y_{str(int(T2-T1))}Y": results
    })

    return out.set_index("Date")

def interpolate_curve(df, new_tenors):

    def _to_years(t):
        t = t.upper().strip()
        if t.endswith("M"):
            return float(t[:-1]) / 12
        elif t.endswith("Y"):
            return float(t[:-1])
        else:
            return float(t)

    # Converte colonne in anni e ordina
    tenor_years = np.array([_to_years(c) for c in df.columns])
    sort_idx = np.argsort(tenor_years)
    df = df.iloc[:, sort_idx]
    tenor_years = tenor_years[sort_idx]

    # Converte i nuovi tenors
    new_years = np.array([_to_years(t) for t in new_tenors])

    # Interpolazione per ogni data
    interpolated_data = {}
    for new_y, new_t in zip(new_years, new_tenors):
        interpolated_data[new_t] = [
            np.interp(new_y, tenor_years, row.values) for _, row in df.iterrows()
        ]

    # Combina tutto in un unico DataFrame
    interpolated_df = pd.DataFrame(interpolated_data, index=df.index)
    full_df = pd.concat([df, interpolated_df], axis=1)

    # Ordina le colonne in base alla maturità
    full_df = full_df[sorted(full_df.columns, key=_to_years)]

    return full_df


def stationarity_test(df):
    df = df.dropna()
    res = pd.DataFrame()
    for c in df.columns:
        tmp = ts.adfuller(df[c])[1]
        tmp = pd.DataFrame(tmp, index=[c], columns=['p_val'])
        res = pd.concat([res, tmp])
    res['below_5%'] = res.apply(lambda x: x < .05)
    return res

### Tickers Generation:

In [ ]:
# tenors to download
tenors_integers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 50]
tenors_str = [str(t) + "Y" for t in tenors_integers]

# tickers generation
tickers_vs_estr = []
tickers_vs_3s6s = []

for t in tenors_str:
    tickers_vs_estr.append(make_basis_ticker("estr3m", t))
    tickers_vs_3s6s.append(make_basis_ticker("3s6s", t))

### Data Download:

In [ ]:
#data download params:
from_ = dt.datetime(2010,1,1)
to_ = dt.datetime.today()
flds = "PX_LAST"
period = "MONTHLY"


In [ ]:
#BASIS Download:
data_vs_estr = bbg_download_clean_df(tickers_vs_estr, flds, from_, to_, period)
data_vs_3s6s = bbg_download_clean_df(tickers_vs_3s6s, flds, from_, to_, period)

## Re Indexing columns:
data_vs_estr.columns = tenors_str
data_vs_3s6s.columns = tenors_str

## FILL NAS
data_vs_estr = data_vs_estr.fillna(method = "ffill")
data_vs_3s6s = data_vs_3s6s.fillna(method = "ffill")

### Curve Interpolation:

In [ ]:
#defining all tenors for 1y to 50y
t_all = [i for i in range(1, 51)]

# Tenors to interpolate:
to_interp = [i for i in t_all if i not in tenors_integers]
to_interp_str = [str(i) + "Y" for i in to_interp]

## Interpolated curve:
data_vs_estr_interp = interpolate_curve(data_vs_estr, to_interp_str)
data_vs_3s6s_interp = interpolate_curve(data_vs_3s6s, to_interp_str)

## switching to percentage instead of bp

data_vs_estr_interp = data_vs_estr_interp/100
data_vs_3s6s_interp = data_vs_3s6s_interp/100

##FILL NAS
data_vs_estr_interp = data_vs_estr_interp.fillna(method = "ffill")
data_vs_3s6s_interp = data_vs_3s6s_interp.fillna(method = "ffill")

### Computing Fwds

In [ ]:
# fwd rates to compute:

# 1y1y, 2y1y, 3y1y, 3y2y, 5y5y, 5y2y, 7y3y, 10y2y, 10y5y, 12y3y, 15y5y, 10y10y, 15y15y 20y10y, 30y20y

fwd_rates_tuples = [
    ("1Y", "2Y"), ("5Y", "10Y"), ("15Y", "20Y")
]

#Compute fwd rates across difference reference rates
fwd_vs_estr = pd.DataFrame()
fwd_vs_3s6s = pd.DataFrame()

for i in fwd_rates_tuples:
    tmp_estr = forward_series_from_timeseries(data_vs_estr_interp, i[0], i[-1], compounding = "annual")
    tmp_3s6s = forward_series_from_timeseries(data_vs_3s6s_interp, i[0], i[-1], compounding = "annual")

    fwd_vs_estr = pd.concat([fwd_vs_estr, tmp_estr], axis = 1)
    fwd_vs_3s6s = pd.concat([fwd_vs_3s6s, tmp_3s6s], axis = 1)

fwd_vs_estr_bp = fwd_vs_estr*100
fwd_vs_3s6s_bp = fwd_vs_3s6s*100

In [ ]:
## Re-indexing to end-of-month:
fwd_vs_3s6s_bp.index = fwd_vs_3s6s_bp.index + pd.offsets.MonthEnd(0)
fwd_vs_estr_bp.index = fwd_vs_estr_bp.index + pd.offsets.MonthEnd(0)

## Features:

### 1) Repo

In [ ]:
### EONIA/ ESTER ADJUSTMENT
transition_dt = dt.datetime(2010, 10, 31)
sprd_adj = 0.082
roll_ma_weeks = 22


### TICKERS:
## O/N: Estro O/N, Eonia O/N, Gc repo O/N
Ticker_Repo_ON = ['ESTRONA Index', 'EONIA Index', 'HEFRGCCE Index']
cols_on = ['ESTR_ON', 'EONIA_ON', 'Gcma_GC']


## DATA Download:
repo_on_df = bbg_download_clean_df(Ticker_Repo_ON, flds, from_, to_, "WEEKLY")
repo_on_df.columns = cols_on


## Adjusting OIS
repo_on_df.loc[repo_on_df.index < transition_dt, 'ESTR_ON'] = (
    repo_on_df.loc[repo_on_df.index < transition_dt, 'EONIA_ON'] + sprd_adj
)


## Dropping Eonia col bc it is now redundant
repo_on_df = repo_on_df.drop('EONIA_ON', axis = 1)
repo_on_df = repo_on_df.dropna()

In [ ]:
### REPO ADJUSTMENT FOR Q/ENDS
res = []
for i in range(len(repo_on_df.index) -5):
    curr = 100*(repo_on_df['Gcma_GC'].iloc[i] - repo_on_df['ESTR_ON'].iloc[i])
    lag = 100*(repo_on_df['Gcma_GC'].iloc[i + 5] - repo_on_df['ESTR_ON'].iloc[i + 5])
    diff = curr-lag

    if i > 0 and abs(diff) > 10:
        curr = res[-1]
    res.append(curr)

## Add to the dataframe
repo_on_df = repo_on_df.iloc[:-5].copy()
repo_on_df['Gcma_GC_vs_OIS'] = res

#smoothing repo O/N with moving average
repo_on_df['Gcma_GC_vs_OIS'] = repo_on_df['Gcma_GC_vs_OIS'].sort_index().rolling(roll_ma_weeks).mean()

repo_on_df = repo_on_df.dropna()

##  Monthly Resampling:
repo_on_df = repo_on_df[['Gcma_GC_vs_OIS']].resample('M').mean()

repo_on_df.index = repo_on_df.index + pd.offsets.MonthEnd(0)

## 2) Rev.Yankee Issuance

In [ ]:
path = "L:estr-bor basis\\Rev_yankee_data.xlsx"

In [ ]:
rev_yankee_df = pd.read_excel(path).set_index('date').dropna()
rev_yankee_df

rev_yankee_feature = rev_yankee_df[['RV_pct_tot']]
rev_yankee_feature.plot()

## 3) Other Features:

In [ ]:
tickers_others = [
    'ITRX XOVER CDSI GEN 5Y Corp', 'SX5E Index', 'ECBLXLTQ Index',
    'EUSS1000 Curncy']

cols = ['iTraxx', 'Eurostoxx50', 'ExLiq', '10s30s']

features = bbg_download_clean_df(tickers_others, flds, from_, to_, period)
features.columns = cols
features = features.dropna()
features = features.sort_values(by = 'date', ascending = True)

##re-indexing at Month-end
features.index = features.index + pd.offsets.MonthEnd(0)

###Adding repo:
features = pd.concat([features, repo_on_df, rev_yankee_feature], axis = 1).fillna(method = 'bfill')

#features.to_clipboard()

In [ ]:
features = features[features.index >= dt.datetime(2015,1,1)]

In [ ]:
database_3s6s = pd.concat([features, fwd_vs_3s6s_bp], axis = 1)
database_estr = pd.concat([features, fwd_vs_estr_bp], axis = 1)

database_3s6s = database_3s6s.dropna()
database_estr = database_estr.dropna()

### Differencing variables:

In [ ]:
m_diff = 12

database_3s6s['Eurostoxx50_ret'] = database_3s6s['Eurostoxx50'].pct_change(m_diff)
database_3s6s['Itraxx_chg'] = database_3s6s['Itraxx'].diff(m_diff)
database_3s6s['ExLiq_chg_bn'] = database_3s6s['ExLiq'].diff(m_diff)/1000
database_3s6s['Gcma_GC_vs_OIS_chg'] = database_3s6s['Gcma_GC_vs_OIS'].diff(m_diff)
database_3s6s['10s30s_chg'] = database_3s6s['10s30s'].diff(m_diff)
database_3s6s['RV_pct_tot_avg'] = database_3s6s['RV_pct_tot'].rolling(m_diff).mean()*100
database_3s6s['ExLiq_tr'] = database_3s6s['ExLiq']/1000000

database_3s6s['Fwd_1Y_1Y_chg'] = database_3s6s['Fwd_1Y_1Y'].diff(m_diff)
database_3s6s['Fwd_5Y_5Y_chg'] = database_3s6s['Fwd_5Y_5Y'].diff(m_diff)
database_3s6s['Fwd_15Y_15Y_chg'] = database_3s6s['Fwd_15Y_15Y'].diff(m_diff)

##dropping NaNs
database_3s6s = database_3s6s.dropna()

#### Estr database:

In [ ]:
database_estr['Eurostoxx50_ret'] = database_estr['Eurostoxx50'].pct_change(m_diff)
database_estr['Itraxx_chg'] = database_estr['Itraxx'].diff(m_diff)
database_estr['ExLiq_chg_bn'] = database_estr['ExLiq'].diff(m_diff)/1000
database_estr['Gcma_GC_vs_OIS_chg'] = database_estr['Gcma_GC_vs_OIS'].diff(m_diff)
database_estr['10s30s_chg'] = database_estr['10s30s'].diff(m_diff)
database_estr['RV_pct_tot_avg'] = database_estr['RV_pct_tot'].rolling(m_diff).mean()*100
database_estr['ExLiq_tr'] = database_estr['ExLiq']/1000000

database_estr['Fwd_1Y_1Y_chg'] = database_estr['Fwd_1Y_1Y'].diff(m_diff)
database_estr['Fwd_5Y_5Y_chg'] = database_estr['Fwd_5Y_5Y'].diff(m_diff)
database_estr['Fwd_15Y_15Y_chg'] = database_estr['Fwd_15Y_15Y'].diff(m_diff)

##dropping NaNs
database_estr = database_estr.dropna()

## Data Analysis:

#### Auxiliary Functions:

In [ ]:
def scatter(df, y_, x_, equation = True, differences = False):
    if differences == True:
        df = check_df_order(df, "ascending")
        df = df.ffill(method = "ffill")
        df = df.diff()
        df = df.dropna()

    reg = linear_model.LinearRegression()
    x = df[x_].values.reshape(-1,1)
    y = df[y_].values.reshape(-1,1)
    reg.fit(x, y)
    alpha, beta = reg.intercept_, reg.coef_[0]
    r_sq = reg.score(x,y)
    scatter = sns.lmplot(x = x_, y = y_, data = df, fit_reg = True)
    props = dict(boxstyle='round', facecolor=sns.color_palette()[0])
    textstr = "R2 = " + str(round(r_sq, 3))
    scatter.ax.text(0, 0, textstr, transform = scatter.ax.transAxes, fontsize = 10, bbox=props)
    return scatter

def OLS_stat(x, y, intercept = False, summary = False):
    if intercept == True:
        x = sm.add_constant(x)
    model = sm.OLS(y, x)
    fit = model.fit()
    params = fit.params
    r_2 = fit.rsquared
    r_2adj = fit.rsquared_adj
    p_vals = fit.pvalues
    std_errors = fit.scale**.5
    if summary == True:
        print(fit.summary())
    if intercept == True:
        coeff = params
        return coeff, p_vals, round(r_2, 5), round(r_2adj, 5), std_errors
    else:
        beta = params[0]
        return beta, p_vals, r_2, r_2adj, std_errors

def fit_model(df, x_vars_list, y_var_list, constant = True, plot_res = False, summary = True, robust_std_err = False):
    X = df[x_vars_list]
    y = df[y_var_list]
    if constant == True:
        X = sm.add_constant(X)
    model = sm.OLS(y, X)
    fit = model.fit()
    if robust_std_err == True:
        fit = model.fit(cov_type='HAC', cov_kwds={'maxlags':1})
    if summary == True:
        print(fit.summary())
    res = pd.DataFrame(fit.fittedvalues, columns = ['y_est'])
    res = pd.concat([res, y], axis = 1)
    res['residuals'] = res[y_var_list[0]] - res['y_est']
    if plot_res == True:
        res[[y_var_list[0], 'y_est']].plot(figsize = (16, 5), title ='fitted vs observed')
        plot_acf(res['residuals'], lags = 50)

    return {
        'results': res,
        'fit': fit,
        'params': fit.params,
        'R2': fit.rsquared,
        'SE': fit.bse
    }

def correlation(df, diff = False):
    fig, ax = plt.subplots(figsize=(15,15))
    if diff == True:
        df = check_df_order(df, 'descending')
        corr = df.diff().corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)
    else:
        corr = df.corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)


# Regression Testing:

## 3s6s Basis:

### 1y1y 3s6s basis:

In [ ]:
euribor3s6s_1y1y_model = fit_model(database_3s6s, ['Itraxx', 'ExLiq_tr'], ['Fwd_1Y_1Y'],
                                   plot_res = True, robust_std_err = False)

### 5y5y 3s6s basis:

In [ ]:
euribor3s6s_5y5y_model = fit_model(database_3s6s, ['10s30s', 'RV_pct_tot_avg'], ['Fwd_5Y_5Y'],
                                   plot_res = True, robust_std_err = False)

### 15y15y 3s6s basis:

In [ ]:
euribor3s6s_15y15y_model = fit_model(database_3s6s, ['10s30s', 'RV_pct_tot_avg'], ['Fwd_15Y_15Y'],
                                     plot_res = True, robust_std_err = False)

## ESTR-BOR Basis:

### 1y1y

In [ ]:
estr_1y1y_model = fit_model(database_estr, ['Itraxx', 'ExLiq_tr'], ['Fwd_1Y_1Y'],
                                   plot_res = True, robust_std_err = False)

### 5y5y

In [ ]:
estr_5y5y_model = fit_model(database_estr,, ['10s30s', 'Itraxx'], ['Fwd_5Y_5Y'],
                                   plot_res = True, robust_std_err = False)

### 15y15y

In [ ]:
estr_15y15y_model = fit_model(database_estr,, ['10s30s', 'Itraxx'], ['Fwd_15Y_15Y'],
                                   plot_res = True, robust_std_err = False)

# Export:

In [ ]:
### Adding z-scores of Residuals:

cols_3s6s = ['1y1y_3s6s_est', 'Fwd_1Y_1Y', '1y1y_3s6s_res',
             '5y5y_3s6s_est', 'Fwd_5Y_5Y', '5y5y_3s6s_res',
             '15y15y_3s6s_est', 'Fwd_15Y_15Y', '15y15y_3s6s_res']

cols_estr = ['1y1y_ee_est', 'Fwd_1Y_1Y', '1y1y_ee_res',
             '5y5y_ee_est', 'Fwd_5Y_5Y', '5y5y_ee_res',
             '15y15y_ee_est', 'Fwd_15Y_15Y', '15y15y_ee_res']


models_3s6s = pd.concat([
    euribor3s6s_1y1y_model_res, euribor3s6s_5y5y_model_res, euribor3s6s_15y15y_model_res],
    axis = 1)

models_ee = pd.concat([
    estr_1y1y_model_res, estr_5y5y_model_res, estr_15y15y_model_res],
    axis = 1)


models_3s6s.columns = cols_3s6s
models_ee.columns = cols_estr

### Adding Z-scores

In [ ]:
### adding z-scores 3s6s

res_cols_3s6s = ['1y1y_3s6s_res', '5y5y_3s6s_res', '15y15y_3s6s_res']
cols_to_add_3s6s = ['1y1y_3s6s_zscore', '5y5y_3s6s_zscore', '15y15y_3s6s_zscore']

z_scores_3s6s = pd.DataFrame()

for col in res_cols_3s6s:
    tmp = models_3s6s[col]
    mean_ = tmp.mean()
    std_ = tmp.std()
    z_score = (tmp - mean_) / std_
    z_scores_3s6s = pd.concat([z_scores_3s6s, z_score], axis=1)

z_scores_3s6s.columns = cols_to_add_3s6s

models_3s6s = pd.concat([models_3s6s, z_scores_3s6s], axis=1)

In [ ]:
### adding z-scores ee

res_cols_ee = ['1y1y_ee_res', '5y5y_ee_res', '15y15y_ee_res']

cols_to_add_ee = ['1y1y_ee_zscore', '5y5y_ee_zscore', '15y15y_ee_zscore']

z_scores_ee = pd.DataFrame()

for col in res_cols_ee:
    tmp = models_ee[col]
    mean_ = tmp.mean()
    std_ = tmp.std()
    z_score = (tmp - mean_) / std_
    z_scores_ee = pd.concat([z_scores_ee, z_score], axis=1)

z_scores_ee.columns = cols_to_add_ee

models_ee = pd.concat([models_ee, z_scores_ee], axis=1)

### Aggregating params:

In [ ]:
### Aggregating parameters:


index_params_order_ee = ['const', 'Itraxx', 'ExLiq_tn', '10s30s', 'R2']
index_params_order_3s6s = ['const', 'Itraxx', 'ExLiq_tn', 'RV_pct_tot_avg', '10s30s', 'R2']

# Coeffs ee
coefficents_ee = pd.concat([estr_1y_1y_params, estr_5y_5y_params, estr_15y15y_params], 1)
coefficents_ee = coefficents_ee.T[index_params_order_ee].T


#Coeffs 3s6s
coefficents_3s6s = pd.concat([euribor3s6s_1y1y_params, euribor3s6s_5y5y_params, euribor3s6s_15y15y_params], 1)
coefficents_3s6s = coefficents_3s6s.T[index_params_order_3s6s].T